In [1]:
# === Cell 1: 导入工具 + 加载 train 数据 ===
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import time
import os

# 路径(notebook 在 notebooks/ 下,所以要 .. )
DATA_RAW = "../data/raw"
DATA_PROCESSED = "../data/processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)

# 加载 train 集
print("加载 train.csv ...")
start = time.time()
train = pd.read_csv(f"{DATA_RAW}/BPC_5core_train.csv")
print(f"✅ {len(train):,} 行,用时 {time.time()-start:.1f} 秒")
print(f"   内存: {train.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
print(train.head(3))

加载 train.csv ...
✅ 5,165,289 行,用时 1.9 秒
   内存: 827.6 MB

                        user_id parent_asin  rating      timestamp
0  AGKASBHYZPGTEPO6LWZPVJWB2BVA  B00V6R3R3S     5.0  1452647102000
1  AGKASBHYZPGTEPO6LWZPVJWB2BVA  B00PA7VMD2     3.0  1452648690000
2  AGKASBHYZPGTEPO6LWZPVJWB2BVA  B00JIIUJ5Q     4.0  1454675735000


In [2]:
# === Cell 2: 构造正样本 + 用户-商品映射字典 ===

# Step 1: 筛选正样本(rating >= 4)
print("Step 1: 筛选正样本(rating >= 4)...")
positives = train[train['rating'] >= 4].copy()
positives['label'] = 1
print(f"  正样本: {len(positives):,} 行 ({len(positives)/len(train)*100:.1f}%)")
print()

# Step 2: 建 user_id → set(已正交互的 items) 字典
# 用于负采样时排除已知正样本,避免"假负样本"
print("Step 2: 建 user → items 字典(用于避免假负样本)...")
start = time.time()
user_to_pos_items = positives.groupby('user_id')['parent_asin'].apply(set).to_dict()
print(f"  完成,用时 {time.time()-start:.1f} 秒")
print(f"  覆盖 {len(user_to_pos_items):,} 个用户")
print()

# Step 3: 商品候选池(用 numpy 数组,采样快)
print("Step 3: 商品候选池...")
all_items_array = positives['parent_asin'].unique()
print(f"  商品总数: {len(all_items_array):,}")
print()

# Step 4: 商品热度(用于流行度负采样)
print("Step 4: 商品热度统计...")
item_pop_counts = positives['parent_asin'].value_counts()  # Series
# 把热度归一化成概率(归一化指数 0.75 是 word2vec 经典做法,缓和长尾)
item_pop_probs = item_pop_counts ** 0.75
item_pop_probs = item_pop_probs / item_pop_probs.sum()

# 创建数组形式,方便 np.random.choice
items_for_pop = item_pop_probs.index.values  # 商品 ID 数组
probs_for_pop = item_pop_probs.values         # 对应概率数组
print(f"  热度概率分布建好,Top 5 热门商品:")
print(item_pop_probs.head(5))

Step 1: 筛选正样本(rating >= 4)...
  正样本: 4,108,857 行 (79.5%)

Step 2: 建 user → items 字典(用于避免假负样本)...
  完成,用时 7.1 秒
  覆盖 718,231 个用户

Step 3: 商品候选池...
  商品总数: 206,246

Step 4: 商品热度统计...
  热度概率分布建好,Top 5 热门商品:
parent_asin
B0BVGHXZJ1    0.000337
B01LSUQSB0    0.000300
B07YT2VKTG    0.000288
B0BS71PXPX    0.000284
B001MA0QY2    0.000278
Name: count, dtype: float64


In [3]:
# === Cell 3: 核心负采样函数 ===

def sample_negatives_for_user(
    user_id, 
    pos_items_set, 
    n_negs=5, 
    pop_ratio=0.5
):
    """
    为一个用户生成 n_negs 个负样本
    
    参数:
        user_id: 用户 ID
        pos_items_set: 该用户的正样本商品集合(用于排除)
        n_negs: 要生成的负样本数(默认 5)
        pop_ratio: 流行度采样的比例(默认 50%)
    
    返回:
        list of (user_id, item_id, 0) 元组
    """
    n_pop = int(n_negs * pop_ratio)     # 流行度负采样个数
    n_random = n_negs - n_pop            # 随机负采样个数
    
    negatives = []
    max_attempts = 50  # 防止死循环(用户买了几乎所有商品的极端情况)
    
    # === 第 1 部分: 随机负采样 ===
    attempts = 0
    while len(negatives) < n_random and attempts < max_attempts:
        # 一次批量抽 n_random 个(比单个抽快很多)
        batch_size = (n_random - len(negatives)) * 2  # 多抽一些,有重复时不用再来
        candidates = np.random.choice(all_items_array, size=batch_size, replace=True)
        for cand in candidates:
            if cand not in pos_items_set and len(negatives) < n_random:
                negatives.append((user_id, cand, 0))
        attempts += 1
    
    # === 第 2 部分: 流行度负采样 ===
    attempts = 0
    target_count = n_random + n_pop
    while len(negatives) < target_count and attempts < max_attempts:
        batch_size = (target_count - len(negatives)) * 2
        candidates = np.random.choice(items_for_pop, size=batch_size, replace=True, p=probs_for_pop)
        for cand in candidates:
            if cand not in pos_items_set and len(negatives) < target_count:
                negatives.append((user_id, cand, 0))
        attempts += 1
    
    return negatives


# 测试: 给第一个用户跑一次
test_user = positives['user_id'].iloc[0]
test_negs = sample_negatives_for_user(
    user_id=test_user,
    pos_items_set=user_to_pos_items[test_user],
    n_negs=5
)
print(f"测试用户 {test_user[:20]}...")
print(f"  正样本商品数: {len(user_to_pos_items[test_user])}")
print(f"  生成的负样本:")
for u, i, l in test_negs:
    print(f"    {i}, label={l}")
print()
print("✅ 函数测试通过")

测试用户 AGKASBHYZPGTEPO6LWZP...
  正样本商品数: 2
  生成的负样本:
    B082S1M1FV, label=0
    B013DRLMOY, label=0
    B0CC56Z8HY, label=0
    B017KUNIBA, label=0
    B07R1XBWY9, label=0

✅ 函数测试通过


In [5]:
# === Cell 4 (向量化版): 批量负采样 ===

N_NEGS = 5
POP_RATIO = 0.5
OVERSAMPLE = 3  # 多抽 3 倍候选,容错冲突

n_pos = len(positives)
n_per_pos_random = int(N_NEGS * (1 - POP_RATIO))   # 每正样本随机负采样数 = 2
n_per_pos_pop = N_NEGS - n_per_pos_random           # 流行度负采样数 = 3 (rounding)
# 实际: 5 * 0.5 = 2.5 → 取 2 随机 + 3 流行度,或 3 随机 + 2 流行度
# 用 2 随机 + 3 流行度

print(f"配置: 每正样本生成 {n_per_pos_random} 随机 + {n_per_pos_pop} 流行度 = {N_NEGS} 个负样本")
print(f"总正样本: {n_pos:,}")
print(f"目标总负样本: {n_pos * N_NEGS:,}")
print()

# === Step 1: 一次性抽所有候选 ===
print("Step 1: 一次性抽样海量候选...")
start = time.time()

n_random_total = n_pos * n_per_pos_random * OVERSAMPLE  # 容错倍数
n_pop_total = n_pos * n_per_pos_pop * OVERSAMPLE

print(f"  随机候选: {n_random_total:,}(含 {OVERSAMPLE}x 容错)")
random_candidates = np.random.choice(all_items_array, size=n_random_total, replace=True)

print(f"  流行度候选: {n_pop_total:,}")
pop_candidates = np.random.choice(items_for_pop, size=n_pop_total, replace=True, p=probs_for_pop)

print(f"  抽样完成,用时 {time.time()-start:.1f} 秒")
print()

# === Step 2: 对每个正样本,从候选里取够 N_NEGS 个不重复的负样本 ===
print("Step 2: 过滤+组装训练数据...")
start = time.time()

all_negs_users = []
all_negs_items = []

random_ptr = 0  # 随机候选的指针
pop_ptr = 0     # 流行度候选的指针

for i in tqdm(range(n_pos), desc="过滤", unit="正样本"):
    user_id = positive_users[i]
    pos_items_set = user_to_pos_items[user_id]
    
    # 抽 n_per_pos_random 个随机负样本
    got = 0
    while got < n_per_pos_random and random_ptr < n_random_total:
        cand = random_candidates[random_ptr]
        random_ptr += 1
        if cand not in pos_items_set:
            all_negs_users.append(user_id)
            all_negs_items.append(cand)
            got += 1
    
    # 抽 n_per_pos_pop 个流行度负样本
    got = 0
    while got < n_per_pos_pop and pop_ptr < n_pop_total:
        cand = pop_candidates[pop_ptr]
        pop_ptr += 1
        if cand not in pos_items_set:
            all_negs_users.append(user_id)
            all_negs_items.append(cand)
            got += 1

elapsed = time.time() - start
print()
print(f"✅ 完成,用时 {elapsed:.1f} 秒")
print(f"   生成负样本: {len(all_negs_users):,} 行")
print(f"   平均速度: {n_pos/elapsed:,.0f} 正样本/秒")
print(f"   候选用了: 随机 {random_ptr:,}/{n_random_total:,}, 流行度 {pop_ptr:,}/{n_pop_total:,}")

配置: 每正样本生成 2 随机 + 3 流行度 = 5 个负样本
总正样本: 4,108,857
目标总负样本: 20,544,285

Step 1: 一次性抽样海量候选...
  随机候选: 24,653,142(含 3x 容错)
  流行度候选: 36,979,713
  抽样完成,用时 5.5 秒

Step 2: 过滤+组装训练数据...


过滤: 100%|███████████████████| 4108857/4108857 [00:05<00:00, 729952.53正样本/s]


✅ 完成,用时 5.6 秒
   生成负样本: 20,544,285 行
   平均速度: 728,551 正样本/秒
   候选用了: 随机 8,219,122/24,653,142, 流行度 12,332,337/36,979,713


In [6]:
# === Cell 5: 拼接正负样本 + 保存到磁盘 ===

# Step 1: 正样本(rating>=4 那些,带 label=1)
print("Step 1: 准备正样本...")
pos_df = positives[['user_id', 'parent_asin']].copy()
pos_df['label'] = 1
print(f"  正样本: {len(pos_df):,} 行")

# Step 2: 负样本(刚才采的,label=0)
print("Step 2: 准备负样本...")
neg_df = pd.DataFrame({
    'user_id': all_negs_users,
    'parent_asin': all_negs_items,
    'label': 0
})
print(f"  负样本: {len(neg_df):,} 行")

# Step 3: 拼接 + 打乱(避免训练时所有正样本在前,负样本在后)
print("Step 3: 拼接 + 打乱...")
train_with_neg = pd.concat([pos_df, neg_df], ignore_index=True)
train_with_neg = train_with_neg.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"  总训练样本: {len(train_with_neg):,} 行")
print(f"  正负比例: {len(pos_df) / len(neg_df):.4f} (≈ 1:5)")
print()

# Step 4: 保存
output_path = f"{DATA_PROCESSED}/train_with_neg.csv"
print(f"Step 4: 保存到 {output_path} ...")
start = time.time()
train_with_neg.to_csv(output_path, index=False)
print(f"  ✅ 保存完成,用时 {time.time()-start:.1f} 秒")
print(f"  文件大小: {os.path.getsize(output_path) / 1024**2:.1f} MB")
print()

# 看一下样本
print("【样本预览(打乱后前 10 行)】")
print(train_with_neg.head(10))
print()
print("【label 分布】")
print(train_with_neg['label'].value_counts())

Step 1: 准备正样本...
  正样本: 4,108,857 行
Step 2: 准备负样本...
  负样本: 20,544,285 行
Step 3: 拼接 + 打乱...
  总训练样本: 24,653,142 行
  正负比例: 0.2000 (≈ 1:5)

Step 4: 保存到 ../data/processed/train_with_neg.csv ...
  ✅ 保存完成,用时 17.3 秒
  文件大小: 987.6 MB

【样本预览(打乱后前 10 行)】
                        user_id parent_asin  label
0  AFX6QX23ONM3KSO4R3B5PWCG6ZWQ  B0B7DXLNF9      0
1  AFABGK73JVSOF733PLSIIQCMIAJA  B00FAZHMNI      0
2  AE5Z4HCOBJ6WIO32RTVBLIND72YA  B07GXFKFDD      0
3  AFJD6PJCU4VB3F5N2SSJGKYUKTOQ  B001D3HWPO      0
4  AHHMWCUBLH72XLSIHJ6KDN4NBOOQ  B003UNAD92      0
5  AFZDSIKYFTU6ZXSFJYTKY6EBHDZQ  B08F2TBTH1      1
6  AFWX42STP4GA3B3ORU66C65YZJ7A  B0094Q6VZE      0
7  AFBPVI74Y2E3DI7IQ6VDUMEGDNCA  B09NN2726V      0
8  AEM3OBLR22FNMMJBM7EUSV3I2H5A  B0BB71N7MN      0
9  AGNQAPOSVHFVJH543JYPC4TSPIAA  B0BS9NTRJ5      0

【label 分布】
label
0    20544285
1     4108857
Name: count, dtype: int64
